# Matching Engine Validation

Deterministic examples covering FIFO priority, price priority, market orders, crossing limits, and partial fills.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
from lob_engine.core.matching_engine import MatchingEngine
from lob_engine.core.orders import Order, OrderStatus, OrderType, Side

engine = MatchingEngine()
engine.process_order(Order('ASK-OLD', Side.SELL, OrderType.LIMIT, 10, 101.0, timestamp=1))
engine.process_order(Order('ASK-NEW', Side.SELL, OrderType.LIMIT, 10, 101.0, timestamp=2))
result = engine.process_order(Order('BUY-MKT', Side.BUY, OrderType.MARKET, 15, timestamp=3))
[(t.passive_order_id, t.quantity) for t in result.trades]

In [ ]:
assert [(t.passive_order_id, t.quantity) for t in result.trades] == [('ASK-OLD', 10), ('ASK-NEW', 5)]

In [ ]:
engine = MatchingEngine()
engine.process_order(Order('ASK-WORSE', Side.SELL, OrderType.LIMIT, 10, 102.0, timestamp=1))
engine.process_order(Order('ASK-BETTER', Side.SELL, OrderType.LIMIT, 10, 101.0, timestamp=2))
result = engine.process_order(Order('BUY-MKT', Side.BUY, OrderType.MARKET, 10, timestamp=3))
[(t.price, t.passive_order_id) for t in result.trades]

In [ ]:
engine = MatchingEngine()
engine.process_order(Order('ASK-1', Side.SELL, OrderType.LIMIT, 5, 100.0, timestamp=1))
result = engine.process_order(Order('BUY-LMT', Side.BUY, OrderType.LIMIT, 8, 101.0, timestamp=2))
print(result.order.status, result.order.filled_quantity, result.order.remaining_quantity)
engine.book.top_n_levels(5)